In [1]:
import sys
sys.path.insert(0, "/projappl/project_2012747/mars/MarS")  # folder that contains market_simulation/
from market_simulation.models.order_model import OrderModel


2026-01-13 13:31:14,914 - /projappl/project_2012747/mars/MarS/market_simulation/__init__.py:15 - INFO - init logging


/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os, glob
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
import wandb

from market_simulation.models.order_model import OrderModel


os.environ["WANDB_NOTEBOOK_NAME"] = "mars_order_model_sweep"

# -------------------------
# Fixed Config
# -------------------------
K = 1024
TRAIN_STRIDE = 16
VAL_STRIDE = 16

BATCH_SIZE = 8
LR = 3e-4
MAX_STEPS = 500 #20_000
EVAL_EVERY = 50
VAL_MAX_BATCHES = 200

USE_AMP = True
AMP_DTYPE = torch.bfloat16  # or torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feat_cols = [f"f{i}" for i in range(15)]

# -------------------------
# Feature prep
# -------------------------
def add_f4(batch):
    t = np.asarray(batch["Time"], dtype=np.int64)
    t_sec = (t // 1_000_000_000).astype(np.int64)
    batch["f4"] = np.clip(t_sec - 34200, 0, 23399).astype(np.int64)
    return batch

def load_parquet_as_cols(path: str):
    ds = load_dataset("parquet", data_files={"data": path})["data"]
    ds = ds.map(add_f4, batched=True, batch_size=200_000, num_proc=1)

    drop_cols = [c for c in ds.column_names if c not in feat_cols]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)

    ds = ds.with_format("numpy")
    cols = {c: ds[c] for c in feat_cols}
    return cols

class SegmentedBatchedWindowIterable(IterableDataset):
    def __init__(self, segments, K, stride, batch_size, random=False, seed=0):
        self.segments = segments
        self.K = K
        self.stride = stride
        self.batch_size = batch_size
        self.random = random
        self.seed = seed

        self.seg_grids = []
        self.seg_weights = []
        for cols in segments:
            N = len(next(iter(cols.values())))
            max_start = N - K
            if max_start < 0:
                grid = np.empty((0,), dtype=np.int64)
            else:
                grid = np.arange(0, max_start + 1, stride, dtype=np.int64)
            self.seg_grids.append(grid)
            self.seg_weights.append(len(grid))

        self.seg_weights = np.asarray(self.seg_weights, dtype=np.float64)
        self.total_starts = self.seg_weights.sum()

    def __iter__(self):
        B = self.batch_size
        K = self.K
        feat_cols_local = feat_cols

        info = torch.utils.data.get_worker_info()
        worker_id = info.id if info else 0
        rng = np.random.default_rng(self.seed + worker_id)

        if self.random:
            if self.total_starts <= 0:
                raise RuntimeError("No segment has enough rows to form a single window (N < K everywhere).")

            probs = self.seg_weights / self.total_starts

            while True:
                seg_ids = rng.choice(len(self.segments), size=B, replace=True, p=probs)

                X = np.empty((B, K, 15), dtype=np.int64)
                for bi, sid in enumerate(seg_ids):
                    grid = self.seg_grids[sid]
                    s = grid[rng.integers(0, len(grid))]
                    cols = self.segments[sid]
                    for fj, c in enumerate(feat_cols_local):
                        X[bi, :, fj] = cols[c][s:s+K]

                yield torch.from_numpy(X).long()

        else:
            for sid, cols in enumerate(self.segments):
                grid = self.seg_grids[sid]
                if len(grid) == 0:
                    continue

                for i in range(0, len(grid), B):
                    s_batch = grid[i:i+B]
                    if len(s_batch) == 0:
                        break

                    X = np.empty((len(s_batch), K, 15), dtype=np.int64)
                    for bi, s in enumerate(s_batch):
                        for fj, c in enumerate(feat_cols_local):
                            X[bi, :, fj] = cols[c][s:s+K]

                    yield torch.from_numpy(X).long()

# -------------------------
# Losses
# -------------------------
def lm_loss_all_positions(logits: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    targets = X[:, :, 0]
    logits_s = logits[:, :-1, :]
    targ_s   = targets[:, 1:]
    return F.cross_entropy(
        logits_s.reshape(-1, logits_s.size(-1)),
        targ_s.reshape(-1),
        reduction="mean",
    )

@torch.no_grad()
def compute_val_loss(model, val_dl) -> float:
    model.eval()
    total = 0.0
    count = 0
    for b, X in enumerate(val_dl, start=1):
        X = X.to(device, non_blocking=True)
        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
            logits = model(X)
            loss = lm_loss_all_positions(logits, X)

        n = X.size(0) * (X.size(1) - 1)
        total += loss.item() * n
        count += n

        if VAL_MAX_BATCHES is not None and b >= VAL_MAX_BATCHES:
            break

    model.train()
    return total / max(1, count)

# -------------------------
# Main train (one wandb run = one grid point)
# -------------------------
def main():
    wandb.login(key="2eec5f6bab880cdbda5c825881bbd45b4b3819d9")
    wandb.init(project="MarS", group="Grid-search", resume=False) # mode="disabled"
    cfg = wandb.config

    # Repro
    seed = int(cfg.get("seed", 123))
    torch.manual_seed(seed)
    np.random.seed(seed)

    # -------------------------
    # Choose training files based on train_fraction
    # -------------------------
    train_files = sorted(glob.glob("../data/features/train_*.parquet"))
    val_files   = sorted(glob.glob("../data/features/test_*.parquet"))

    if len(train_files) == 0:
        raise RuntimeError("No train_*.parquet found in ../data/features/")
    if len(val_files) == 0:
        raise RuntimeError("No test_*.parquet found in ../data/features/")

    train_fraction = float(cfg.get("train_fraction", 1.0))
    if train_fraction < 1.0:
        # deterministic subset: shuffle with seed, take half (or fraction)
        rng = np.random.default_rng(seed)
        idx = np.arange(len(train_files))
        rng.shuffle(idx)
        keep = max(1, int(round(train_fraction * len(train_files))))
        train_files = [train_files[i] for i in idx[:keep]]

    wandb.log({
        "n_train_files": len(train_files),
        "n_val_files": len(val_files),
        "train_fraction_effective": len(train_files),
    }, step=0)

    # Load segments
    train_segments = [load_parquet_as_cols(p) for p in train_files]
    val_segments   = [load_parquet_as_cols(p) for p in val_files]

    train_iterable = SegmentedBatchedWindowIterable(
        train_segments, K=K, stride=TRAIN_STRIDE, batch_size=BATCH_SIZE,
        random=True, seed=seed
    )
    val_iterable = SegmentedBatchedWindowIterable(
        val_segments, K=K, stride=VAL_STRIDE, batch_size=BATCH_SIZE,
        random=False, seed=seed
    )

    train_dl = DataLoader(train_iterable, batch_size=None, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_iterable,   batch_size=None, num_workers=2, pin_memory=True)

    # -------------------------
    # Model size grid
    # -------------------------
    model_size = cfg.get("model_size", "base")
    if model_size == "base":
        EMB_DIM, NUM_LAYERS, NUM_HEADS = 64, 2, 4   # ~your current run (~5M-ish)
    elif model_size == "small":
        # smaller model: significantly fewer params
        EMB_DIM, NUM_LAYERS, NUM_HEADS = 32, 1, 4
    else:
        raise ValueError(f"Unknown model_size={model_size}")

    model = OrderModel(
        emb_dim=EMB_DIM,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        num_max_orders=K,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    wandb.config.update({
        "emb_dim": EMB_DIM,
        "num_layers": NUM_LAYERS,
        "num_heads": NUM_HEADS,
        "n_params": n_params,
    }, allow_val_change=True)

    print(f"Model params: {n_params/1e6:.2f}M")
    opt = torch.optim.AdamW(model.parameters(), lr=LR)

    use_scaler = (USE_AMP and device.type == "cuda" and AMP_DTYPE == torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    # -------------------------
    # Train loop
    # -------------------------
    model.train()
    train_it = iter(train_dl)

    pbar = tqdm(range(1, MAX_STEPS + 1), desc=f"train [{model_size}] frac={train_fraction}")
    for step in pbar:
        X = next(train_it)
        X = X.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
            logits = model(X)
            loss = lm_loss_all_positions(logits, X)

        if use_scaler:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()

        train_loss = float(loss.item())
        pbar.set_postfix(train_loss=f"{train_loss:.4f}")

        wandb.log({"train_loss": train_loss}, step=step)

        if step % EVAL_EVERY == 0:
            val_loss = compute_val_loss(model, val_dl)
            print(f"\nstep {step:6d} | val_loss {val_loss:.4f}\n")
            wandb.log({"val_loss": val_loss}, step=step)

    # final eval
    val_loss = compute_val_loss(model, val_dl)
    wandb.log({"val_loss_final": val_loss}, step=MAX_STEPS)

if __name__ == "__main__":
    main()


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


Model params: 10.18M


train [base] frac=1.0:  10%|█         | 52/500 [01:25<56:01,  7.50s/it, train_loss=9.4688]  


step     50 | val_loss 10.0069



train [base] frac=1.0:  20%|██        | 102/500 [02:49<44:54,  6.77s/it, train_loss=7.6563]  


step    100 | val_loss 8.7626



train [base] frac=1.0:  30%|██▉       | 148/500 [03:25<05:13,  1.12it/s, train_loss=6.1169]